# Résoudre `6_shortage` avec OR-Tools
Sixième jeu de données : 15 000 modèles pour seulement **5 000 datasets**. *« Where are the files? »* La demande de données vaut 28 fois l'offre, et l'énergie limite aussi. **Tout est rare à la fois.**

Ce notebook montre :
1. que les sources **doublent** le score ;
2. comment **diagnostiquer** une perte : ce qui reste dans les réserves, puis le gaspillage ;
3. qu'une **marge** sur l'offre totale, calibrée sur ce gaspillage, récupère une bonne partie de la perte.

**Version 3.** Ce notebook n'implémente plus rien lui-même : il appelle le package `ia_responsable` (dossier `src/`). Le récit et les expériences sont ceux de la version précédente ; le code est celui que teste `make test` et qu'utilise `make solve`. Les fonctions sont commentées dans les modules indiqués entre parenthèses.

**Durée d'exécution** : environ 35 minutes. Sur ce dataset, SCIP met plusieurs minutes à résoudre la seule relaxation linéaire de départ : avec 5 minutes, selon la machine, il peut ne pas avoir fini et rendre une très mauvaise solution. On lui laisse donc 15 minutes par résolution.

## 1. Charger l'instance

In [1]:
from pathlib import Path
import json, time, random
from collections import Counter
import pandas as pd

from ia_responsable import load
from ia_responsable.cli import official_score
from ia_responsable.fill import fill, give, new_pools, no_full_models
from ia_responsable.model import solve
from ia_responsable.packing import REFERENCE_ORDERS, noisy_orders, pack
from ia_responsable.pipeline import plan_value, run
from ia_responsable.repair import State, complete, repair

pd.set_option("display.width", 150)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
inst = load(ROOT / "datasets" / "6_shortage.json")
WINNER = 669_798


def score(plan):
    """Score officiel (test_solution.py). Lève une erreur si le plan est invalide."""
    s, valid, message = official_score(ROOT, inst.text, plan)
    assert valid, message
    return s


def solver_row(info, sel):
    return dict(statut=info.status, objectif=info.objective, borne=round(info.bound, 1), écart_prouvé=round(info.gap, 1),
                secondes=round(info.seconds), pleins=len(sel.full), demi=len(sel.half), sources=len(sel.sources))


def failed_full(sel, plan):
    """Modèles choisis à pleine valeur qui n'ont pas eu leurs données libres."""
    data = {}
    for d, m in plan["dataMappings"]:
        data.setdefault(m, []).append(d)
    return [m for m in sel.full if m not in data or any(inst.copyrighted[d] for d in data[m])]


def save_if_better(plan):
    out = ROOT / "solutions" / f"{inst.name}_main.json"
    new = score(plan)
    current = official_score(ROOT, inst.text, json.loads(out.read_text()))[0] if out.exists() else 0
    if new > current:
        out.write_text(json.dumps(plan))
        print(f"{out.name} mis à jour : {current:,} -> {new:,}")
    else:
        print(f"{out.name} inchangé ({current:,}) ; ce notebook a obtenu {new:,}")


print(f"{inst.name} : {len(inst.value)} modèles, {len(inst.size)} datasets | gagnant du concours : {WINNER:,}")

6_shortage : 15000 modèles, 5000 datasets | gagnant du concours : 669,798


## 2. Ce qui change

In [2]:
demand = {t: sum(low.get(t, 0) for low in inst.lower.values()) for t in "ntic"}
print(f"énergie de tous les modèles : {sum(inst.energy.values()) / inst.energy_cap:.2f} × le plafond")
print(f"sources possibles : {len(inst.mono)} | modèles impossibles : {len(inst.impossible)}")
mono_lower = pd.Series([inst.lower[s][t] for s, t in inst.mono.items()])
print(f"besoin d'une source : médiane {mono_lower.median():,.0f}, 10 % demandent moins de {mono_lower.quantile(0.1):,.0f}")
pd.DataFrame({"offre_totale": inst.total_supply, "offre_libre": inst.free_supply,
              "demande_sur_offre": {t: round(demand[t] / inst.total_supply[t], 1) for t in "ntic"}})

énergie de tous les modèles : 7.57 × le plafond
sources possibles : 5841 | modèles impossibles : 272
besoin d'une source : médiane 29,157, 10 % demandent moins de 5,935


,offre_totale,offre_libre,demande_sur_offre
n,6459411,3794759,27.4
t,6087321,3611587,28.6
i,6264424,3702287,28.1
c,6277125,3742205,28.7


**Lecture** : chaque type offre environ 6 millions d'unités, de quoi remplir à peine 300 besoins. Une cible dont **tous** les besoins sont couverts par des sources n'a besoin d'**aucun** dataset : quand les données manquent à ce point, c'est le levier principal.

## 3. Contrôle préalable
Un modèle dont un besoin ne peut pas être rempli en données libres seules est interdit à pleine valeur (`fill.no_full_models`).

In [3]:
no_full = no_full_models(inst)
print("interdits à pleine valeur :", sorted(no_full))

interdits à pleine valeur : [255, 9946]


## 4. Les sources doublent le score

In [4]:
runs = {}
sel_old, info_old = solve(inst, seconds=120, possible_sources=False, no_full=no_full)
runs["sources : besoins impossibles seulement"] = solver_row(info_old, sel_old)
sel, info = solve(inst, seconds=900, no_full=no_full)
runs["sources partout"] = solver_row(info, sel)
pd.DataFrame(runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,351198,351275.9,77.9,122,1570,458,78
sources partout,FEASIBLE,670872,670946.4,74.4,903,992,1250,1317


**Lecture** : avec des sources partout, le score **double**, et la sélection dépasse le gagnant sur le papier. Plus d'un millier de modèles servent de sources.

## 5. Attribuer, compléter, réparer

In [5]:
def search(sel, n_noisy=200, n_finish=5):
    """Deux ordres de référence + n_noisy ordres bruités ; complète et répare les n_finish meilleures attributions."""
    orders = dict(REFERENCE_ORDERS) | noisy_orders(inst, n_noisy, seed=0, spread=0.3)
    packed = []
    for name, key in orders.items():
        plan, n_failed = pack(inst, sel, key)
        packed.append((plan_value(inst, plan), name, n_failed, plan))
    packed.sort(key=lambda r: -r[0])
    rows, best = [], (0, None, None, None)
    for value0, name, n_failed, plan in packed[:n_finish]:
        state = State(inst, plan, no_full)
        added = complete(state)
        n = repair(state, max_partners=None)
        s = score(state.plan())
        rows.append(dict(ordre=name, ratés=n_failed, après_attribution=value0, ajouts=added, réparations=n, score=s))
        if s > best[0]:
            best = (s, name, state.plan(), plan)
    return best, pd.DataFrame(rows).set_index("ordre")

t0 = time.time()
best_a, table_a = search(sel)
print(f"{time.time() - t0:.0f} s | sélection : {info.objective:,} | meilleur plan : {best_a[0]:,}")
table_a

12 s | sélection : 670,872 | meilleur plan : 667,879


,ratés,après_attribution,ajouts,réparations,score
ordre,,,,,
bruité 143,19,667198,12,5,667879
bruité 3,18,667144,12,3,667814
bruité 193,18,667144,11,1,667852
bruité 54,18,667140,13,4,667778
bruité 42,19,667133,13,6,667758


**Lecture** : on perd plusieurs milliers de points entre la sélection du solveur et le plan réel, même avec le meilleur des 200 ordres. Il faut comprendre pourquoi.

## 6. Diagnostiquer la perte
Première question : **quelle ressource manque ?** On regarde ce qui reste dans les réserves après la meilleure attribution brute.

In [6]:
raw_plan = best_a[3]
used = Counter()
for d, m in raw_plan["dataMappings"]:
    used[(inst.dtype[d], inst.copyrighted[d])] += inst.size[d]
pd.DataFrame({"libre_restant": {t: inst.free_supply[t] - used[(t, 0)] for t in "ntic"},
              "sous_droits_restant": {t: inst.total_supply[t] - inst.free_supply[t] - used[(t, 1)] for t in "ntic"}})

,libre_restant,sous_droits_restant
n,6598,3003
t,829,980
i,11941,3326
c,980,17104


Les deux réserves sont **presque vides** : c'est l'**offre totale** qui coince.

Deuxième question : **où sont passées les données ?** Le modèle compte, pour chaque besoin, exactement sa borne basse ; `fill` remplit chaque besoin **un peu au-dessus**. Mesurons ce gaspillage.

In [7]:
use_by_need = Counter()
for d, m in best_a[2]["dataMappings"]:
    use_by_need[(m, inst.dtype[d])] += inst.size[d]
waste = Counter()
for (m, t), total in use_by_need.items():
    waste[t] += total - inst.lower[m][t]
pd.DataFrame({"gaspillage": waste, "part_de_l_offre": {t: f"{waste[t] / inst.total_supply[t]:.2%}" for t in "ntic"}})

,gaspillage,part_de_l_offre
i,53106,0.85%
t,42071,0.69%
n,50683,0.78%
c,45424,0.72%


**Lecture** : le gaspillage représente environ **0,8 % de l'offre** de chaque type. Le modèle prévoit donc plus de données qu'il n'en existe réellement, et les derniers modèles servis échouent.

Deux remèdes « intuitifs » essayés pendant la préparation, qui **ne marchent pas** :
- une marge sur les **données libres** : moins bien, ce n'est pas la ressource qui manque ;
- un `fill` qui gaspille moins (choisir, pour finir, la combinaison la plus juste) : **beaucoup** moins bien (environ 640 000). Les paires consomment de petits datasets qui manquent ensuite à d'autres modèles.

Le remède qui marche : dire au modèle qu'il ne peut utiliser qu'environ **98,5 %** de l'offre totale (`total_margin`).

In [8]:
sel_m, info_m = solve(inst, seconds=900, total_margin=0.985, no_full=no_full)
runs["sources partout, marge 1,5 % sur l'offre totale"] = solver_row(info_m, sel_m)
display(pd.DataFrame(runs).T)
t0 = time.time()
best_b, table_b = search(sel_m)
print(f"{time.time() - t0:.0f} s | meilleur plan : {best_b[0]:,}")
table_b

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,351198,351275.9,77.9,122,1570,458,78
sources partout,FEASIBLE,670872,670946.4,74.4,903,992,1250,1317
"sources partout, marge 1,5 % sur l'offre totale",FEASIBLE,668749,668857.0,108.0,903,990,1242,1303


15 s | meilleur plan : 668,822


,ratés,après_attribution,ajouts,réparations,score
ordre,,,,,
plus contraints,4,667884,14,2,668822
bruité 156,14,667248,18,2,668290
bruité 40,13,667233,20,2,668297
bruité 19,15,667193,21,1,668342
bruité 110,16,667177,21,3,668207


**Lecture** : sur le papier, la sélection avec marge vaut moins ; une fois les datasets attribués, elle donne **plus**. **Une solution un peu moins bonne sur le papier peut être bien meilleure en réalité**, si elle est plus facile à réaliser.

## 7. Comparaison

In [9]:
best = max(best_a, best_b, key=lambda b: b[0])
summary = pd.DataFrame({"score": {"plan sans marge": best_a[0], "plan avec marge 1,5 %": best_b[0], "gagnant du concours": WINNER,
                                  "borne du solveur (sans marge)": round(info.bound),
                                  "sans sources sur besoins remplissables": info_old.objective}})
summary["écart_au_gagnant"] = summary.score - WINNER
summary["part_du_gagnant"] = (summary.score / WINNER).map("{:.2%}".format)
display(summary)
save_if_better(best[2])

,score,écart_au_gagnant,part_du_gagnant
plan sans marge,667879,-1919,99.71%
"plan avec marge 1,5 %",668822,-976,99.85%
gagnant du concours,669798,0,100.00%
borne du solveur (sans marge),670946,1148,100.17%
sans sources sur besoins remplissables,351198,-318600,52.43%


6_shortage_main.json inchangé (668,822) ; ce notebook a obtenu 668,822


## À retenir
1. **Quand tout manque, les sources deviennent l'essentiel** : une cible entièrement couverte n'a besoin d'aucun dataset.
2. **Diagnostiquer avant de corriger** : ce qui reste dans les réserves, puis le gaspillage, ont désigné la vraie cause.
3. **Une marge calibrée sur une mesure**, pas choisie au hasard.
4. **Optimiser le modèle n'est pas optimiser la réalité.**
5. **Contrôler les hypothèses du modèle** : le contrôle préalable écarte les modèles impossibles à servir en données libres.